# HR Employee Statistical Analysis

**Repository**: `hr-employee-statistical-analysis`

Nghiên cứu các yếu tố ảnh hưởng đến quyết định nghỉ việc của nhân viên (Employee Attrition) bằng mô hình Hồi quy Logistic (Logistic Regression).

## Mục tiêu

1. Phân tích dữ liệu nhân viên nghỉ việc (EDA)
2. Xây dựng mô hình Logistic Regression để dự đoán nghỉ việc
3. Đánh giá hiệu năng mô hình
4. Phân tích Odds Ratio để diễn giải các yếu tố ảnh hưởng

## Dataset

- File: `data/WA_Fn-UseC_-HR-Employee-Attrition.csv`
- 1,470 quan sát
- Biến mục tiêu: `Attrition` (1 = churn, 0 = không churn)


In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(".")  # thư mục gốc: C:\Workspace\Latex\hr-employee-statistical-analysis
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from hr_employee.analysis.eda import compute_eda_summary, churn_rate_by_category
from hr_employee.config import get_default_paths
from hr_employee.data.io import DatasetSpec, load_churn_dataset

## 1. Import Libraries và Setup


In [ ]:
import sys
from pathlib import Path
import json
from dataclasses import asdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, HTML

# Import modules from src/hr_employee
from hr_employee.analysis.eda import compute_eda_summary, churn_rate_by_category
from hr_employee.config import get_default_paths
from hr_employee.data.io import DatasetSpec, load_churn_dataset
from hr_employee.evaluation.metrics import compute_metrics
from hr_employee.model.logistic import LogisticSpec
from hr_employee.preprocessing.features import FeatureSpec
from hr_employee.preprocessing.pipeline import split_xy
from hr_employee.reporting.odds_ratio import plot_odds_ratio_forest, write_odds_ratio_analysis
from hr_employee.reporting.project_report import write_project_report
from hr_employee.stats.logit_odds_ratio import fit_logit_and_odds_ratio
from hr_employee.training.pipeline import build_sklearn_pipeline, stratified_split
from hr_employee.utils.fs import ensure_dir
from hr_employee.visualization.plots import (
    plot_churn_rate_bar,
    plot_confusion_matrix,
    plot_numeric_distribution,
    plot_roc_curve,
)

# Setup paths
paths = get_default_paths()
ensure_dir(paths.figures_dir)
ensure_dir(paths.outputs_dir)

# Numeric columns to visualize
EDA_NUMERIC_HIGHLIGHTS = ("Age", "MonthlyIncome", "YearsAtCompany", "DistanceFromHome")

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"✓ Data directory: {paths.data_dir}")
print(f"✓ Outputs directory: {paths.outputs_dir}")
print(f"✓ Figures directory: {paths.figures_dir}")


## 2. Load Data


In [ ]:
# Load dataset
dataset_spec = DatasetSpec(csv_path=paths.data_dir / "WA_Fn-UseC_-HR-Employee-Attrition.csv")
raw_df = load_churn_dataset(dataset_spec)

print(f"Dataset shape: {raw_df.shape}")
print(f"\nColumns: {list(raw_df.columns)}")
print(f"\nFirst few rows:")
display(raw_df.head())

print(f"\nData types:")
print(raw_df.dtypes)

print(f"\nMissing values:")
print(raw_df.isnull().sum())


## 3. Exploratory Data Analysis (EDA)


In [ ]:
# Feature specification
feature_spec = FeatureSpec()

# Compute EDA summary
summary = compute_eda_summary(raw_df, feature_spec)

# Calculate churn counts
n_churn = int(summary.n_rows * summary.churn_rate)
n_non_churn = summary.n_rows - n_churn

print("=" * 60)
print("EDA Summary")
print("=" * 60)
print(f"Total samples: {summary.n_rows:,}")
print(f"Churn rate: {summary.churn_rate:.2%}")
print(f"Non-churn: {n_non_churn:,} ({1 - summary.churn_rate:.2%})")
print(f"Churn: {n_churn:,} ({summary.churn_rate:.2%})")
print(f"\nCategorical columns: {feature_spec.categorical_columns}")
print(f"Numeric columns: {feature_spec.numeric_columns}")


In [ ]:
# Churn rate by categorical variables
for cat in feature_spec.categorical_columns:
    df_rate = churn_rate_by_category(
        raw_df, category_column=cat, feature_spec=feature_spec
    )
    print(f"\nChurn rate by {cat}:")
    display(df_rate)

    # Plot inline
    plt.figure(figsize=(8, 4))
    ax = sns.barplot(data=df_rate, x=cat, y="churn_rate")
    ax.set_title(f"Churn rate by {cat}")
    ax.set_xlabel(cat)
    ax.set_ylabel("Churn rate")
    plt.tight_layout()
    plt.show()

    # Also save to file
    plot_churn_rate_bar(
        df_rate,
        category_column=cat,
        out_path=paths.figures_dir / f"churn_rate_by_{cat}.png",
    )


In [ ]:
# Distribution plots for numeric variables
for col in EDA_NUMERIC_HIGHLIGHTS:
    print(f"\nDistribution of {col} by churn:")

    # Plot inline
    plt.figure(figsize=(8, 4))
    ax = sns.kdeplot(
        data=raw_df,
        x=col,
        hue=feature_spec.target_column,
        common_norm=False,
        fill=True,
        alpha=0.35,
    )
    ax.set_title(f"Distribution of {col} by {feature_spec.target_column}")
    plt.tight_layout()
    plt.show()

    # Also save to file
    plot_numeric_distribution(
        raw_df,
        numeric_column=col,
        target_column=feature_spec.target_column,
        out_path=paths.figures_dir / f"dist_{col}_by_churn.png",
    )


## 4. Preprocessing và Train/Test Split


In [ ]:
# Split X and y
x, y = split_xy(raw_df, feature_spec)

print(f"Features (X) shape: {x.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nTarget proportion:")
print(y.value_counts(normalize=True))


In [ ]:
# Stratified train/test split
split = stratified_split(x, y, test_size=0.2, random_state=42)

train_churn = np.mean(split.y_train)
test_churn = np.mean(split.y_test)

print("Train/Test Split:")
print(f"  Train: {len(split.x_train):,} samples (churn: {train_churn:.2%})")
print(f"  Test:  {len(split.x_test):,} samples (churn: {test_churn:.2%})")
print(f"\n✓ Split preserves churn rate in both sets")


## 5. Train Logistic Regression Model


## 5.1 Train toàn bộ pipeline (giống script Python)

Cell này gọi trực tiếp hàm `main()` trong `scripts/run_pipeline.py`:
- Load data
- EDA + sinh hình
- Train Logistic Regression
- Đánh giá (metrics, ROC, confusion matrix)
- Fit Logit (statsmodels) + Odds Ratio
- Sinh báo cáo `PROJECT_REPORT.md` và `odds_ratio_analysis.txt`

Điều này cho thấy **notebook đang dùng đúng code của dự án**, không tự nhập số tay.


In [ ]:
# Run full Python pipeline (giống chạy: python scripts/run_pipeline.py)
from scripts.run_pipeline import main as run_full_pipeline

run_full_pipeline()


In [ ]:
# Build and train pipeline
logistic_spec = LogisticSpec(class_weight="balanced", random_state=42)
pipeline = build_sklearn_pipeline(feature_spec, logistic_spec)

print("Model Configuration:")
print(f"  Solver: {logistic_spec.solver}")
print(f"  Penalty: {logistic_spec.penalty}")
print(f"  C: {logistic_spec.c}")
print(f"  Class weight: {logistic_spec.class_weight}")
print("\n=== Training model ===")

# Train
pipeline.fit(split.x_train, split.y_train)

# Get feature count after preprocessing
n_features = pipeline.named_steps["preprocess"].transform(split.x_train[:1]).shape[1]
print(f"✓ Model trained on {n_features} features")

# Show internal iterations of solver (n_iter_)
model = pipeline.named_steps["model"]
try:
    print(f"✓ Solver iterations (per class): {model.n_iter_}")
except AttributeError:
    pass

# Quick evaluation right after training
print("\n=== Quick Evaluation (Train/Test) ===")

y_prob_train = pipeline.predict_proba(split.x_train)[:, 1]
y_prob_test = pipeline.predict_proba(split.x_test)[:, 1]

metrics_train = compute_metrics(
    np.asarray(split.y_train), np.asarray(y_prob_train), threshold=0.5
)
metrics_test = compute_metrics(
    np.asarray(split.y_test), np.asarray(y_prob_test), threshold=0.5
)

print("Train:")
print(f"  ROC AUC  : {metrics_train.roc_auc:.3f}")
print(f"  Accuracy : {metrics_train.accuracy:.3f}")
print(f"  Precision: {metrics_train.precision:.3f}")
print(f"  Recall   : {metrics_train.recall:.3f}")
print(f"  F1       : {metrics_train.f1:.3f}")

print("\nTest:")
print(f"  ROC AUC  : {metrics_test.roc_auc:.3f}")
print(f"  Accuracy : {metrics_test.accuracy:.3f}")
print(f"  Precision: {metrics_test.precision:.3f}")
print(f"  Recall   : {metrics_test.recall:.3f}")
print(f"  F1       : {metrics_test.f1:.3f}")

print("\nConfusion Matrix (Test):")
print(f"  TN (True Negative):  {metrics_test.confusion_matrix[0,0]}")
print(f"  FP (False Positive): {metrics_test.confusion_matrix[0,1]}")
print(f"  FN (False Negative): {metrics_test.confusion_matrix[1,0]}")
print(f"  TP (True Positive):  {metrics_test.confusion_matrix[1,1]}")


## 6. Model Evaluation


In [ ]:
# Predictions
y_prob_train = pipeline.predict_proba(split.x_train)[:, 1]
y_prob_test = pipeline.predict_proba(split.x_test)[:, 1]

# Compute metrics
metrics_train = compute_metrics(
    np.asarray(split.y_train), np.asarray(y_prob_train), threshold=0.5
)

metrics_test = compute_metrics(
    np.asarray(split.y_test), np.asarray(y_prob_test), threshold=0.5
)

# Display metrics
print("=" * 60)
print("Model Performance Metrics")
print("=" * 60)

metrics_df = pd.DataFrame({
    "Train": [
        metrics_train.roc_auc,
        metrics_train.accuracy,
        metrics_train.precision,
        metrics_train.recall,
        metrics_train.f1,
    ],
    "Test": [
        metrics_test.roc_auc,
        metrics_test.accuracy,
        metrics_test.precision,
        metrics_test.recall,
        metrics_test.f1,
    ],
}, index=["ROC AUC", "Accuracy", "Precision", "Recall", "F1"])

display(metrics_df.round(4))

print(f"\nConfusion Matrix (Test):")
print(f"  TN (True Negative):  {metrics_test.confusion_matrix[0,0]}")
print(f"  FP (False Positive): {metrics_test.confusion_matrix[0,1]}")
print(f"  FN (False Negative): {metrics_test.confusion_matrix[1,0]}")
print(f"  TP (True Positive):  {metrics_test.confusion_matrix[1,1]}")


In [ ]:
# Plot ROC Curve inline
from sklearn.metrics import RocCurveDisplay

plt.figure(figsize=(6, 5))
RocCurveDisplay.from_predictions(np.asarray(split.y_test), y_prob_test)
plt.title("ROC Curve")
plt.tight_layout()
plt.show()

print(f"\nROC AUC = {metrics_test.roc_auc:.4f}")

# Also save to file
plot_roc_curve(
    np.asarray(split.y_test), y_prob_test, paths.figures_dir / "roc_curve.png"
)


In [ ]:
# Plot Confusion Matrix inline
plt.figure(figsize=(5, 4))
ax = sns.heatmap(metrics_test.confusion_matrix, annot=True, fmt="d", cmap="Blues")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

# Also save to file
plot_confusion_matrix(
    metrics_test.confusion_matrix, paths.figures_dir / "confusion_matrix.png"
)


## 7. Odds Ratio Analysis (Statsmodels)


In [ ]:
# Fit Logit model and compute Odds Ratios
design_matrix, result, or_df = fit_logit_and_odds_ratio(raw_df, feature_spec)

print("=" * 60)
print("Odds Ratio Table")
print("=" * 60)
display(or_df)

# Save to CSV
or_df.to_csv(paths.outputs_dir / "odds_ratio_table.csv", index=False)
print(f"\n✓ Saved to: {paths.outputs_dir / 'odds_ratio_table.csv'}")


In [ ]:
# Display statsmodels summary
print("=" * 60)
print("Statsmodels Logit Summary")
print("=" * 60)
print(result.summary())

# Save summary
(paths.outputs_dir / "logit_summary.txt").write_text(
    str(result.summary()), encoding="utf-8"
)
print(f"\n✓ Saved to: {paths.outputs_dir / 'logit_summary.txt'}")


In [ ]:
# Forest Plot for Odds Ratios (inline)
or_df_plot = or_df[or_df["feature"] != "const"].sort_values("p_value", ascending=True).copy()
or_df_plot["significant"] = (or_df_plot["p_value"] < 0.05) & ~(
    (or_df_plot["ci_lower"] <= 1.0) & (1.0 <= or_df_plot["ci_upper"])
)

plt.figure(figsize=(10, 8))
ax = plt.gca()

y_pos = list(range(len(or_df_plot)))
colors = ["#2ecc71" if bool(sig) else "#95a5a6" for sig in or_df_plot["significant"]]

# CI bars
for i, (_, row) in enumerate(or_df_plot.iterrows()):
    ax.plot(
        [float(row["ci_lower"]), float(row["ci_upper"])],
        [i, i],
        color=colors[i],
        linewidth=2,
        alpha=0.8,
    )

# OR points
ax.scatter(
    or_df_plot["odds_ratio"].astype(float),
    y_pos,
    s=90,
    c=colors,
    edgecolors="black",
    linewidths=1.2,
    zorder=3,
    alpha=0.95,
)

ax.axvline(
    x=1.0,
    color="red",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label="OR = 1 (No effect)",
)

ax.set_yticks(y_pos)
ax.set_yticklabels(or_df_plot["feature"].astype(str))
ax.set_xlabel("Odds Ratio (95% CI)")
ax.set_title("Forest Plot: Odds Ratio các yếu tố ảnh hưởng nghỉ việc\n(Xanh = p<0.05, Xám = không ý nghĩa)")
ax.set_xscale("log")
ax.grid(axis="x", alpha=0.3, linestyle=":")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

# Also save to file
plot_odds_ratio_forest(
    paths.outputs_dir / "odds_ratio_table.csv",
    paths.figures_dir / "forest_plot_odds_ratio.png",
)


In [ ]:
# Key findings from Odds Ratio
print("=" * 60)
print("Key Findings from Odds Ratio Analysis")
print("=" * 60)

# Risk factors (OR > 1)
risk_factors = or_df[or_df["odds_ratio"] > 1].sort_values("odds_ratio", ascending=False)
print("\n🔴 Risk Factors (OR > 1):")
for _, row in risk_factors.iterrows():
    print(f"  • {row['feature']}: OR = {row['odds_ratio']:.4f} (95% CI: [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}])")

# Protective factors (OR < 1)
protective_factors = or_df[or_df["odds_ratio"] < 1].sort_values("odds_ratio", ascending=True)
print("\n🟢 Protective Factors (OR < 1):")
for _, row in protective_factors.iterrows():
    print(f"  • {row['feature']}: OR = {row['odds_ratio']:.4f} (95% CI: [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}])")


## 8. Save Results và Generate Reports


In [ ]:
# Save metrics
import joblib

metrics_dict = {
    "roc_auc": metrics_test.roc_auc,
    "accuracy": metrics_test.accuracy,
    "precision": metrics_test.precision,
    "recall": metrics_test.recall,
    "f1": metrics_test.f1,
    "confusion_matrix": metrics_test.confusion_matrix.tolist(),
}

with open(paths.outputs_dir / "logistic_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)

print(f"✓ Metrics saved to: {paths.outputs_dir / 'logistic_metrics.json'}")

# Save model
joblib.dump(pipeline, paths.outputs_dir / "logistic_pipeline.joblib")
print(f"✓ Model saved to: {paths.outputs_dir / 'logistic_pipeline.joblib'}")

# Save EDA summary
with open(paths.outputs_dir / "eda_summary.json", "w", encoding="utf-8") as f:
    json.dump(asdict(summary), f, indent=2, ensure_ascii=False)
print(f"✓ EDA summary saved to: {paths.outputs_dir / 'eda_summary.json'}")


In [ ]:
# Generate Odds Ratio analysis report
write_odds_ratio_analysis(
    paths.outputs_dir / "odds_ratio_table.csv",
    paths.outputs_dir / "odds_ratio_analysis.txt",
)
print(f"✓ Odds Ratio analysis saved to: {paths.outputs_dir / 'odds_ratio_analysis.txt'}")

# Generate project report
write_project_report(paths)
print(f"✓ Project report saved to: PROJECT_REPORT.md")


## 9. Tổng kết (Summary)

### Kết quả chính:

1. **Hiệu năng mô hình**:
   - ROC AUC, Recall, Accuracy (xem ở trên)
   - Mô hình phát hiện được nguy cơ nghỉ việc

2. **Yếu tố ảnh hưởng nổi bật**:
   - Xem bảng Odds Ratio ở trên

3. **Outputs đã tạo**:
   - Metrics: `outputs/logistic_metrics.json`
   - Model: `outputs/logistic_pipeline.joblib`
   - Odds Ratios: `outputs/odds_ratio_table.csv`
   - Figures: `figures/*.png`
   - Reports: `PROJECT_REPORT.md`, `outputs/odds_ratio_analysis.txt`

### Giải pháp/Khuyến nghị:

1. Tăng tỷ lệ active member
2. Chương trình giữ chân theo nhóm tuổi
3. Phân tích sâu mức độ nghỉ việc do overtime và Sales
4. Tối ưu decision threshold theo chi phí/lợi ích
